# Paper #80 Implementation: Waves in the Lower Solar Atmosphere / 태양 저대기의 파동

Based on Jess, Jafarzadeh, Keys et al. (2023), LRSP, DOI: 10.1007/s41116-022-00035-6.

1. Acoustic cutoff vs height
2. k-ω diagnostic diagram
3. MHD wave dispersion in β regimes
4. Synthetic 3-min umbral oscillation
5. Mode conversion at β=1
6. DKIST micro-jet observation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

GAMMA = 5/3
G_SUN = 2.74e4
K_B = 1.38e-16
M_H = 1.67e-24

## 1. Acoustic Cutoff ω_c = c_s / (2H) / 음향 차단 주파수

$\omega_c = c_s/(2H)$ where $c_s = \sqrt{\gamma p/\rho}$ and $H = k_B T/(\mu m_H g)$.

In [ ]:
def acoustic_cutoff_mHz(T):
    """Acoustic cutoff frequency in mHz.

    Args:
        T: Temperature (K).

    Returns:
        ω_c/(2π) in mHz.
    """
    H = K_B * T / (0.5 * M_H * G_SUN)
    c_s = np.sqrt(GAMMA * K_B * T / (0.5 * M_H))
    omega_c = c_s / (2 * H)
    return omega_c / (2 * np.pi) / 1e-3

z_km = np.linspace(0, 2500, 100)
T_profile = 6000 - 3000 * (z_km / 500) * np.exp(-(z_km / 500) ** 2) + 2000 * (z_km > 1500)
T_profile = np.maximum(T_profile, 4000)

plt.figure()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(z_km, T_profile / 1000)
ax1.set_xlabel('Height (km)')
ax1.set_ylabel('T (kK)')
ax1.set_title('VAL-like T profile')
ax1.grid(alpha=0.3)

ax2.plot(z_km, acoustic_cutoff_mHz(T_profile))
ax2.axhline(5.5, color='r', linestyle='--', label='5.5 mHz (3-min)')
ax2.axhline(3.3, color='b', linestyle='--', label='3.3 mHz (5-min)')
ax2.set_xlabel('Height (km)')
ax2.set_ylabel('ω_c/(2π) (mHz)')
ax2.set_title('Acoustic cutoff frequency / 음향 차단')
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. k-ω Diagnostic Diagram / k-ω 진단 다이어그램

p-mode ridges: $\omega^2 = c_s^2 k^2 + \omega_c^2$

In [ ]:
k = np.linspace(0.1, 3, 200)
c_s = 7  # Mm/s
omega_c = 2 * np.pi * 5.5e-3

plt.figure()
for n in range(6):
    omega = np.sqrt((n + 0.5) * c_s * k * omega_c)
    plt.plot(k, omega / (2 * np.pi) * 1e3, label=f'p_{n}')
plt.axhline(5.5, color='k', linestyle='--', label='Cutoff')
plt.xlabel('Horizontal k (Mm⁻¹)')
plt.ylabel('ν (mHz)')
plt.title('p-mode diagnostic diagram / p-mode 진단 다이어그램')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. MHD Wave Dispersion in β Regimes / MHD 파동 분산

Fast, slow, Alfvén modes in different plasma β.

In [ ]:
def mhd_speeds(c_s, v_A):
    """Fast and slow magneto-acoustic speeds."""
    c_plus = np.sqrt(0.5 * (c_s ** 2 + v_A ** 2 + np.sqrt((c_s ** 2 + v_A ** 2) ** 2 - 4 * c_s ** 2 * v_A ** 2)))
    c_minus = np.sqrt(0.5 * (c_s ** 2 + v_A ** 2 - np.sqrt((c_s ** 2 + v_A ** 2) ** 2 - 4 * c_s ** 2 * v_A ** 2)))
    return c_plus, c_minus

beta_range = np.logspace(-2, 2, 200)
v_A_const = 10
c_s_vals = v_A_const * np.sqrt(beta_range / 2)

c_fast, c_slow = mhd_speeds(c_s_vals, v_A_const)

plt.figure()
plt.loglog(beta_range, c_fast, label='Fast mode')
plt.loglog(beta_range, c_slow, label='Slow mode')
plt.loglog(beta_range, np.ones_like(beta_range) * v_A_const, '--', label='Alfvén')
plt.axvline(1, color='r', linestyle=':', label='β=1')
plt.xlabel('Plasma β')
plt.ylabel('Phase speed (km/s)')
plt.title('MHD mode speeds vs β / MHD 모드 속도')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

## 4. 3-min Umbral Oscillation / 3분 암부 진동

In [ ]:
t = np.linspace(0, 900, 2000)
wave = 0.8 * np.sin(2 * np.pi * t / 180) + 0.3 * np.sin(2 * np.pi * t / 300)
shock_train = 0.5 * np.exp(-((t % 180 - 50) / 8) ** 2)
signal = wave + shock_train

plt.figure()
plt.plot(t, signal)
plt.xlabel('Time (s)')
plt.ylabel('Intensity (arb)')
plt.title('3-min umbral oscillation with shocks / 3분 암부 진동')
plt.grid(alpha=0.3)
plt.show()

from scipy.signal import periodogram
f, P = periodogram(signal - np.mean(signal), fs=1.0 / (t[1] - t[0]))
plt.figure()
plt.semilogx(f * 1000, P)
plt.axvline(1000 / 180, color='r', linestyle='--', label='180s peak')
plt.xlabel('Frequency (mHz)')
plt.ylabel('Power')
plt.title('Power spectrum / 파워 스펙트럼')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

## 5. Mode Conversion at β=1 / β=1에서 모드 변환

In [ ]:
z = np.linspace(0, 2000, 300)
beta = np.exp(-(z - 1000) / 200)
T_ratio = 1 / (1 + np.exp(-(np.log(beta))))

plt.figure()
plt.semilogy(z, beta)
plt.axhline(1, color='r', linestyle='--')
plt.xlabel('Height (km)')
plt.ylabel('Plasma β')
plt.title('β profile with β=1 layer / β=1 층 포함 프로파일')
plt.grid(alpha=0.3, which='both')
plt.show()

print('At β=1 (equipartition layer): fast↔slow conversion, acoustic→magnetic')

## 6. DKIST Micro-jet Observation / DKIST 마이크로제트 관측

In [ ]:
t_obs = np.linspace(0, 300, 600)
jet = 80 * np.exp(-((t_obs - 100) / 30) ** 2) + 0.5 * np.random.randn(len(t_obs))

plt.figure()
plt.plot(t_obs, jet)
plt.xlabel('Time (s)')
plt.ylabel('Doppler velocity (km/s)')
plt.title('DKIST synthetic micro-jet / DKIST 합성 마이크로제트')
plt.grid(alpha=0.3)
plt.show()

## Summary / 요약
1. Cutoff 3.3 mHz photosphere, rising chromosphere
2. p-mode ridges visible up to ν_ac
3. Fast/slow/Alfvén separate in different β
4. Umbral 3-min oscillation dominant
5. β=1 layer drives mode conversion
6. DKIST resolves sub-arcsec micro-jets

### References
- Jess, D. B. et al. (2023). *Waves in the Lower Solar Atmosphere*. LRSP 20, 1.